In [1]:
%load_ext autoreload
%autoreload 2

from __future__ import absolute_import, division, print_function
import torch
from trainer_endoda3 import Trainer
from options_endoda3 import MonodepthOptions


/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/third_party/EndoDAC/models/backbones/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/third_party/EndoDAC/models/backbones/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/third_party/EndoDAC/models/backbones/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [2]:
# Minimal options for testing
options = MonodepthOptions()
args = [
    '--batch_size', '2',
    '--num_workers', '1',
    '--of_samples',
    '--of_samples_num', '10',
    '--frame_ids', '0', '-1', '1',
    '--dataset', 'endovis',
    '--data_path', '/mnt/nct-zfs/TCO-All/SharedDatasets/SCARED_Images_Resized/',
    '--log_dir', '/tmp/endoda_debug',
    '--compute_metrics',
    '--depth_model_type', 'depthanything3',
    '--endoda3_model_config', '/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/networks/configs/endo-da3-all-wowrapper.yaml',
    '--pose_model_type', 'da3_internal',
    '--k_model_type', 'da3_internal',
    '--of_model_type', 'raft',
    # '--learn_intrinsics',
 
    # '--endoda3_model_config', '/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/networks/configs/endo-da3-depth-wowrapper.yaml'
]
opts = options.parse_notebook(args)

# Initialize trainer
trainer = Trainer(opts)
print(f"Trainer initialized on {trainer.device}")


Loading depth model setting from config: /mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/networks/configs/endo-da3-all-wowrapper.yaml
[INFO ] using MLP layer as FFN

Loading pretrained weights from depth-anything/da3-base
[INFO ] using MLP layer as FFN

depth_model state dict info:
  Missing keys: 48
    Key prefixes (up to 3 levels): ['backbone.blocks.0', 'backbone.blocks.1', 'backbone.blocks.10', 'backbone.blocks.11', 'backbone.blocks.2', 'backbone.blocks.3', 'backbone.blocks.4', 'backbone.blocks.5', 'backbone.blocks.6', 'backbone.blocks.7', 'backbone.blocks.8', 'backbone.blocks.9']
  Unexpected keys: 0
Successfully loaded pretrained weights from depth-anything/da3-base for depth net.

Training model named:
   1219_0140
Models and tensorboard events files are saved to:
   /tmp/endoda_debug
Training is using:
   cuda
Overfitting mode: using 10 Trn samples
Overfitting mode: using 10 Val samples
Overfitting mode: using 10 Test samples
Loading trajectories for 1 unique folders...
S

/mnt/cluster/environments/jinjingxu/pkg/envs/cu12/lib/python3.11/site-packages/torch/functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647348947/work/aten/src/ATen/native/TensorShape.cpp:3595.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [4]:
# Get sample batch
trainer.step = 0
trainer.set_train()
train_iter = iter(trainer.train_loader)
inputs = next(train_iter)

# Forward pass
outputs, losses = trainer.process_batch(inputs)

print("Forward pass completed!")
print(f"Output keys: {len(outputs)} keys")
print(f"Loss: {losses['loss'].item():.6f}")
print(f"Loss components: {list(losses.keys())}")


Resizing input from 256x320 to 224x280
Resizing input from 256x320 to 224x280
Resizing input from 256x320 to 224x280
Forward pass completed!
Output keys: 128 keys
Loss: 0.147026
Loss components: ['loss/0', 'loss/1', 'loss/2', 'loss/3', 'loss']


In [ ]:
# Compute losses explicitly
losses = trainer.compute_losses(inputs, outputs)

print("Loss computation:")
for key, val in losses.items():
    if isinstance(val, torch.Tensor):
        print(f"  {key}: {val.item():.6f}")
    else:
        print(f"  {key}: {val}")


In [ ]:
# Get validation batch (has GT depth and poses)
trainer.set_eval()
val_iter = iter(trainer.val_loader)
val_inputs = next(val_iter)

# Forward pass
with torch.no_grad():
    val_outputs, val_losses = trainer.process_batch(val_inputs)

# Compute depth metrics
from utils.metrics import compute_depth_metrics, compute_pose_metrics

depth_metrics = compute_depth_metrics(val_inputs, val_outputs)
pose_metrics = compute_pose_metrics(val_inputs, val_outputs, opts.frame_ids)

print("Depth Metrics:")
if depth_metrics:
    for key, val in depth_metrics.items():
        print(f"  {key}: {val:.6f}")
else:
    print("  No depth metrics (GT depth not available)")

print("\nPose Metrics:")
if pose_metrics:
    for key, val in pose_metrics.items():
        print(f"  {key}: {val:.6f}")
else:
    print("  No pose metrics (GT poses not available)")


In [ ]:
# Full training step
trainer.set_train()
train_iter = iter(trainer.train_loader)
inputs = next(train_iter)

# Forward
outputs, losses = trainer.process_batch(inputs)

# Backward
trainer.model_optimizer.zero_grad()
losses["loss"].backward()
trainer.model_optimizer.step()

print(f"Training step completed! Loss: {losses['loss'].item():.6f}")
